## v9 — Base(non-lag) + Seq(damped-lag) 블렌딩 (runnable, ASCII filename)

- 저장 파일: `submission_v9_blend.csv`


In [4]:
# NOTE: This notebook is identical in logic to the intended v9 blend notebook,
# but uses an ASCII filename to avoid Windows Unicode path quirks.

import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

# ============================================================
# Tuning knobs
# ============================================================
N_FOLDS = 5
SEED = 42
USE_LOG_TARGET = True

EMA_ALPHA = 0.75
LAG_NOISE_MULT = 0.35
LAG_CLIP_Q = 0.97
BETA_BLEND = 0.75


def _print_progress(msg):
    sys.stdout.write('\r' + msg)
    sys.stdout.flush()

def elapsed(start):
    s = int(time.time() - start)
    return f"{s//60}m {s%60:02d}s"

def section(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")


class LGBProgressCallback:
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold
        self.n_folds = n_folds
        self.total = total_rounds
        self.log_every = log_every
        self.start = time.time()
        self.best = float('inf')
        self.best_it = 0
    def __call__(self, env):
        it = env.iteration + 1
        mae = env.evaluation_result_list[0][2]
        if mae < self.best:
            self.best = mae
            self.best_it = it
        if it % self.log_every == 0 or it == self.total:
            filled = int(30 * it / self.total)
            bar = '█'*filled + '░'*(30-filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val {mae:.4f}  best {self.best:.4f}@{self.best_it}  {elapsed(self.start)}"
            )


class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold
        self.n_folds = n_folds
        self.total = total_rounds
        self.log_every = log_every
        self.start = time.time()
        self.best = float('inf')
        self.best_it = 0
    def after_iteration(self, model, epoch, evals_log):
        it = epoch + 1
        try:
            mae = list(evals_log.values())[0]['mae'][-1]
        except Exception:
            return False
        if mae < self.best:
            self.best = mae
            self.best_it = it
        if it % self.log_every == 0:
            filled = int(30 * it / self.total)
            bar = '█'*filled + '░'*(30-filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val {mae:.4f}  best {self.best:.4f}@{self.best_it}  {elapsed(self.start)}"
            )
        return False


def cat_verbose_step(n):
    return max(1, n // 5)


def _resolve_data_dir() -> str:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / 'data'
        if d.is_dir() and (d / 'train.csv').is_file():
            return str(d)
    raise FileNotFoundError('data/train.csv 를 찾을 수 없습니다. 프로젝트 루트에서 실행하세요.')


def to_train_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y


def from_train_pred(p):
    return np.expm1(p) if USE_LOG_TARGET else p


# 1) Load
path = _resolve_data_dir()
project_root = str(Path(path).resolve().parent)
print(f"▶ data dir: {path}")

TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

train = pd.read_csv(os.path.join(path, 'train.csv'))
test  = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))


# 2) Preprocess (same as v8)
def handle_missing_values(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    cols_to_fix = [c for c in df.columns if df[c].isnull().any() and c not in (ID_COLS + [TARGET])]
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].ffill()
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].bfill()
    df[cols_to_fix] = df[cols_to_fix].fillna(df[cols_to_fix].median())
    return df

def add_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    df['timeslot']          = df.groupby('scenario_id').cumcount()
    df['robot_efficiency']  = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density']     = df['robot_total']  / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure']    = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)
    if 'congestion_score' in df.columns:
        df['risk_index']  = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    roll_cols = ['order_per_station', 'congestion_score', 'pack_utilization', 'avg_trip_distance', 'low_battery_ratio']
    for col in roll_cols:
        if col not in df.columns:
            continue
        grp = df.groupby('scenario_id')[col]
        df[f'{col}_roll3_mean'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'{col}_roll5_mean'] = grp.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        df[f'{col}_roll3_std']  = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))
    return df

def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    df = handle_missing_values(df)
    df = add_features(df)
    if 'layout_type' in df.columns:
        df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df

train = preprocess_all(train, layout)
test  = preprocess_all(test, layout)

# 3) Target encoding
TE_COLS = [c for c in ['layout_id', 'timeslot', 'layout_type', 'shift_hour', 'day_of_week'] if c in train.columns]
SMOOTHING = 20
kf_te = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']
for col in TE_COLS:
    te_col = f'{col}_te'
    train[te_col] = np.nan
    global_mean = train[TARGET].mean()
    for tr_idx, val_idx in kf_te.split(train, train[TARGET], groups=groups_te):
        tr_df = train.iloc[tr_idx]
        stats = tr_df.groupby(col)[TARGET].agg(['mean', 'count'])
        smooth = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        train.loc[val_idx, te_col] = train.iloc[val_idx][col].map(smooth).fillna(global_mean)
    stats_full = train.groupby(col)[TARGET].agg(['mean', 'count'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + SMOOTHING * global_mean) / (stats_full['count'] + SMOOTHING)
    test[te_col] = test[col].map(smooth_full).fillna(global_mean)

train_global_mean = float(train[TARGET].mean())
train_global_std  = float(train[TARGET].std())
LAG_NOISE_STD = float(train_global_std * LAG_NOISE_MULT)
LAG_CLIP_HI = float(np.percentile(train[TARGET].dropna(), 100 * LAG_CLIP_Q))

DAMPED_LAG_COLS = ['lag1_log','lag2_log','lag3_log','lag1_clip','lag2_clip','lag3_clip','lag_diff_12','lag_diff_23','lag_mean3']
for c in DAMPED_LAG_COLS:
    test[c] = 0.0

def compute_damped_lags(dst_df, raw_lag1, raw_lag2, raw_lag3, clip_hi):
    df = dst_df
    df['lag1_log']  = np.log1p(np.maximum(raw_lag1, 0))
    df['lag2_log']  = np.log1p(np.maximum(raw_lag2, 0))
    df['lag3_log']  = np.log1p(np.maximum(raw_lag3, 0))
    df['lag1_clip'] = np.clip(raw_lag1, 0, clip_hi)
    df['lag2_clip'] = np.clip(raw_lag2, 0, clip_hi)
    df['lag3_clip'] = np.clip(raw_lag3, 0, clip_hi)
    df['lag_diff_12'] = raw_lag1 - raw_lag2
    df['lag_diff_23'] = raw_lag2 - raw_lag3
    df['lag_mean3'] = (raw_lag1 + raw_lag2 + raw_lag3) / 3.0
    return df

def add_true_damped_lags(df, clip_hi, noise_std=0.0):
    out = df.sort_values(['scenario_id','ID']).copy()
    raws = {}
    for lag in (1,2,3):
        r = out.groupby('scenario_id')[TARGET].shift(lag)
        scen_mean = out.groupby('scenario_id')[TARGET].transform('mean')
        r = r.fillna(scen_mean)
        if noise_std>0:
            rng = np.random.RandomState(SEED+lag)
            r = r + rng.normal(0, noise_std, size=len(r))
        raws[lag] = r.values
    out = compute_damped_lags(out, raws[1], raws[2], raws[3], clip_hi)
    return out

base_feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
feature_cols_base = base_feature_cols
feature_cols_seq  = base_feature_cols + DAMPED_LAG_COLS

y_all = to_train_target(train[TARGET].values)
groups = train['scenario_id'].values
kf = GroupKFold(n_splits=N_FOLDS)

def sequential_predict_seq_model(model, frame, feature_cols, gm, clip_hi, alpha, predict_train_space):
    va = frame.copy()
    for c in DAMPED_LAG_COLS:
        va[c] = 0.0
    pred = pd.Series(np.nan, index=va.index, dtype=np.float64)
    ema  = pd.Series(np.nan, index=va.index, dtype=np.float64)
    max_slot = int(va['timeslot'].max())
    for slot in range(max_slot+1):
        slot_mask = va['timeslot'].to_numpy()==slot
        slot_idx  = va.index[slot_mask]
        raw_lags = {}
        for lag in (1,2,3):
            prev_slot = slot-lag
            if prev_slot>=0:
                prev_mask = va['timeslot'].to_numpy()==prev_slot
                prev_df = va.loc[prev_mask,['scenario_id']].copy()
                prev_df['ema'] = ema.loc[va.index[prev_mask]].to_numpy()
                scen_to_ema = prev_df.groupby('scenario_id',sort=False)['ema'].last()
                raw_lags[lag] = va.loc[slot_idx,'scenario_id'].map(scen_to_ema).fillna(gm).values
            else:
                raw_lags[lag] = np.full(len(slot_idx), gm)
        tmp = compute_damped_lags(va.loc[slot_idx].copy(), raw_lags[1], raw_lags[2], raw_lags[3], clip_hi)
        for c in DAMPED_LAG_COLS:
            va.loc[slot_idx, c] = tmp[c].values
        X_slot = va.loc[slot_idx, feature_cols]
        p_train = predict_train_space(model, X_slot)
        raw_pred = from_train_pred(np.asarray(p_train, dtype=np.float64))
        pred.loc[slot_idx] = raw_pred
        if slot==0:
            ema.loc[slot_idx] = raw_pred
        else:
            prev1_mask = va['timeslot'].to_numpy()==(slot-1)
            prev1_df = va.loc[prev1_mask,['scenario_id']].copy()
            prev1_df['ema'] = ema.loc[va.index[prev1_mask]].to_numpy()
            scen_prev = prev1_df.groupby('scenario_id',sort=False)['ema'].last()
            prev_ema = va.loc[slot_idx,'scenario_id'].map(scen_prev).fillna(gm).values
            ema.loc[slot_idx] = alpha*raw_pred + (1-alpha)*prev_ema
    return pred

lgb_params = dict(n_estimators=8000, learning_rate=0.01, max_depth=10, num_leaves=255,
                  min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                  reg_alpha=0.2, reg_lambda=2.0, random_state=SEED, verbose=-1)

oofA = np.zeros(len(train)); oofB = np.zeros(len(train))
modelsA=[]; modelsB=[]

section('Fit base/seq LGB only (fast runnable)')
for fold,(tr_idx,va_idx) in enumerate(kf.split(train, y_all, groups=groups),1):
    X_tr = train.iloc[tr_idx][feature_cols_base]
    X_va = train.iloc[va_idx][feature_cols_base]
    y_tr = to_train_target(train.iloc[tr_idx][TARGET].values)
    y_va = to_train_target(train.iloc[va_idx][TARGET].values)
    mA = lgb.LGBMRegressor(**lgb_params)
    mA.fit(X_tr,y_tr,eval_set=[(X_va,y_va)],eval_metric='mae',callbacks=[lgb.early_stopping(200, verbose=False)])
    oofA[va_idx] = from_train_pred(mA.predict(X_va))
    modelsA.append(mA)

    tr_raw = train.iloc[tr_idx].copy(); va_raw=train.iloc[va_idx].copy(); gm=float(tr_raw[TARGET].mean())
    tr_fit = add_true_damped_lags(tr_raw, LAG_CLIP_HI, noise_std=LAG_NOISE_STD)
    va_es  = add_true_damped_lags(va_raw, LAG_CLIP_HI, noise_std=0.0)
    X_tr2 = tr_fit[feature_cols_seq]; X_va2=va_es[feature_cols_seq]
    y_tr2 = to_train_target(tr_fit[TARGET].values); y_va2=to_train_target(va_es[TARGET].values)
    mB = lgb.LGBMRegressor(**lgb_params)
    mB.fit(X_tr2,y_tr2,eval_set=[(X_va2,y_va2)],eval_metric='mae',callbacks=[lgb.early_stopping(200, verbose=False)])
    seq_pred = sequential_predict_seq_model(mB, va_raw, feature_cols_seq, gm, LAG_CLIP_HI, EMA_ALPHA, lambda mdl,X: mdl.predict(X))
    oofB[va_idx] = seq_pred.values
    modelsB.append(mB)

def _mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def _best_beta(y_true, base_pred, seq_pred, grid):
    best_beta = None
    best_mae = float('inf')
    for b in grid:
        p = (1 - b) * base_pred + b * seq_pred
        m = _mae(y_true, p)
        if m < best_mae:
            best_mae = m
            best_beta = float(b)
    return best_beta, float(best_mae)


maeA = _mae(train[TARGET], oofA)
maeB = _mae(train[TARGET], oofB)

# β 자동 탐색 (OOF 최소화)
BETA_GRID = np.round(np.linspace(0.0, 1.0, 21), 2)
best_beta, best_mae = _best_beta(train[TARGET].values, oofA, oofB, BETA_GRID)
print(f"[beta-search] best_beta={best_beta}  best_OOF_MAE={best_mae:.6f}")

# 현재 설정 β로도 출력
beta_use = float(BETA_BLEND)
oofBlend = (1 - beta_use) * oofA + beta_use * oofB
maeBlend = _mae(train[TARGET], oofBlend)
print(f"OOF base: {maeA:.6f}  OOF seq: {maeB:.6f}  OOF blend(beta={beta_use}): {maeBlend:.6f}")

# 기본 권장: seq가 base보다 나쁘면 beta를 best_beta로 자동 적용해도 됨
# (원하면 아래 한 줄을 켜서 제출도 best_beta로 자동 생성 가능)
# beta_use = best_beta

# Test predict
X_test_base = test[feature_cols_base]
pA = np.mean([from_train_pred(m.predict(X_test_base)) for m in modelsA], axis=0)

test2 = test.sort_values(['scenario_id','ID']).reset_index(drop=True)
test2['timeslot'] = test2.groupby('scenario_id').cumcount()
max_slot = int(test2['timeslot'].max())
for c in DAMPED_LAG_COLS: test2[c]=0.0
pB = np.zeros(len(test2)); ema_buf=np.full(len(test2), train_global_mean)
for slot in range(max_slot+1):
    slot_mask = (test2['timeslot'].values==slot)
    slot_idx = np.where(slot_mask)[0]
    raw_lags={}
    for lag in (1,2,3):
        prev=slot-lag
        if prev>=0:
            prev_mask=(test2['timeslot'].values==prev)
            prev_df=test2.loc[prev_mask,['scenario_id']].copy(); prev_df['ema']=ema_buf[prev_mask]
            scen=prev_df.groupby('scenario_id',sort=False)['ema'].last()
            raw_lags[lag]=test2.loc[slot_mask,'scenario_id'].map(scen).fillna(train_global_mean).values
        else:
            raw_lags[lag]=np.full(len(slot_idx), train_global_mean)
    tmp=compute_damped_lags(test2.loc[slot_mask].copy(), raw_lags[1], raw_lags[2], raw_lags[3], LAG_CLIP_HI)
    for c in DAMPED_LAG_COLS: test2.loc[slot_mask,c]=tmp[c].values
    X_slot=test2.loc[slot_mask, feature_cols_seq]
    pred_slot=np.mean([from_train_pred(m.predict(X_slot)) for m in modelsB], axis=0)
    pB[slot_idx]=pred_slot
    if slot==0:
        ema_buf[slot_idx]=pred_slot
    else:
        prev1_mask=(test2['timeslot'].values==(slot-1))
        prev_df=test2.loc[prev1_mask,['scenario_id']].copy(); prev_df['ema']=ema_buf[prev1_mask]
        scen=prev_df.groupby('scenario_id',sort=False)['ema'].last()
        prev_ema=test2.loc[slot_mask,'scenario_id'].map(scen).fillna(train_global_mean).values
        ema_buf[slot_idx]=EMA_ALPHA*pred_slot + (1-EMA_ALPHA)*prev_ema

pBlend = (1-BETA_BLEND)*pA + BETA_BLEND*pB
sub = pd.DataFrame({'ID': test2['ID'], TARGET: pBlend})
save_path = os.path.join(project_root, 'submission_v9_blend.csv')
sub.to_csv(save_path, index=False)
print('saved ->', save_path)


▶ data dir: C:\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\data

  Fit base/seq LGB only (fast runnable)
[beta-search] best_beta=0.0  best_OOF_MAE=8.821197
OOF base: 8.821197  OOF seq: 9.926864  OOF blend(beta=0.75): 9.455231
saved -> C:\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\submission_v9_blend.csv
